In [1]:
!python -m pip install pyyaml==5.1
import sys, os, distutils.core
# Note: This is a faster way to install detectron2 in Colab, but it does not include all functionalities (e.g. compiled operators).
# See https://detectron2.readthedocs.io/tutorials/install.html for full installation instructions
!git clone 'https://github.com/facebookresearch/detectron2'
dist = distutils.core.run_setup("./detectron2/setup.py")
!python -m pip install {' '.join([f"'{x}'" for x in dist.install_requires])}
sys.path.insert(0, os.path.abspath('./detectron2'))

# Properly install detectron2. (Please do not install twice in both ways)
# !python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.2/274.2 kB 6.2 MB/s eta 0:00:00a 0:00:01
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
Cloning into 'detectron2'...
remote: Enumerating objects: 15943, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 15943 (delta 5), reused 4 (delta 4), pack-reused 15931 (from 3)
Receiving objects: 100% (15943/15943), 6.71 MiB | 21.88 MiB/s, done.
Resolving deltas: 100% (11334/11334), done.
Ignoring dataclasses: markers 'python_version < "3.7"' don't match you

In [2]:
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import random
import cv2
from detectron2.structures import BoxMode
from detectron2.data import DatasetCatalog,MetadataCatalog
from detectron2.utils.visualizer import Visualizer
from detectron2.engine import DefaultTrainer
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.data import build_detection_train_loader

NumExpr defaulting to 4 threads.


In [3]:
def rle(rl,h,w):
    mask=np.zeros(h*w,dtype=np.uint8)
    rl=rl.split()
    st=np.array(rl[0::2],dtype=int)-1
    lent=np.array(rl[1::2],dtype=int)
    for s,l in zip(st,lent):
        mask[s:s+l]=1
    return mask.reshape((h,w),order='F')

In [4]:
def box(mask):
    ys,xs=np.where(mask==1)
    xm=xs.min()
    xma=xs.max()
    ym=ys.min()
    yma=ys.max()
    return [xm,ym,xma,yma]

In [5]:
def dictt(dire,csvp):
    df=pd.read_csv(csvp)
    grp=df.groupby('ImageId')
    dd=[]
    for idx,(id,rs) in enumerate(grp):
        if rs['EncodedPixels'].isna().all():
            continue
        fp=os.path.join(dire,id)
        img=cv2.imread(fp)
        h,w=img.shape[:2]
        rec={
            'file_name':fp,
            'image_id':idx,
            'height':h,
            'width':w,
            'annotations':[]
        }
        for _,r in rs.iterrows():
            rl=r['EncodedPixels']
            if pd.isna(rl):
                continue
            mk=rle(rl,h,w)
            bbox=box(mk)
            ann={
                'bbox':bbox,
                'bbox_mode':BoxMode.XYXY_ABS,
                'segmentation':mk,
                'category_id':0
            }
            rec['annotations'].append(ann)
        dd.append(rec)
    return dd

In [6]:
DatasetCatalog.register('train',lambda:dictt('/kaggle/input/airbus-ship-detection-train-set-70/train_v3/train_v3/Images','/kaggle/input/airbus-ship-detection-train-set-70/train_ship_segmentations_v3.csv'))

In [7]:
MetadataCatalog.get('train').set(thing_classes=['ship'])

namespace(name='train', thing_classes=['ship'])

In [8]:
cfg=get_cfg()

In [9]:
cfg.merge_from_file(model_zoo.get_config_file('COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml'))

In [10]:
cfg.OUTPUT_DIR='/kaggle/working/output'

In [11]:
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

In [12]:
cfg.DATASETS.TRAIN=('train',)
cfg.DATASETS.TEST=()

In [13]:
cfg.MODEL.ROI_HEADS.NUM_CLASSES=1

In [14]:
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")

In [15]:
cfg.MODEL.ANCHOR_GENERATOR.SIZES = [[16, 32, 64, 128]]
cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [[0.5, 1.0, 2.0]]

In [16]:
cfg.SOLVER.IMS_PER_BATCH = 4
cfg.SOLVER.BASE_LR = 0.00025
cfg.SOLVER.MAX_ITER = 1000
cfg.SOLVER.STEPS = []   

In [ ]:
mod=DefaultTrainer(cfg)
mod.resume_or_load(resume=False)

Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res2): Sequential(
        (0): Bottlene

In [ ]:
mod.train()